# Clustering Model Comparison

Put `train.csv` and `test.csv` in the same directory as this notebook.

- No `TARGET_COLUMN` — clustering is unsupervised, there are no labels.
- No preprocessing is performed.
- Models are fit on **train.csv only**.
- `test.csv` is used only for final evaluation.
- Since there's no ground truth, evaluation uses **Silhouette Score** instead of accuracy/R2 (higher is better, range -1 to 1).
- The final cell reports Silhouette Score for every model on both train and test.


In [ ]:
# Common imports and data loading
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, AgglomerativeClustering, BisectingKMeans
from sklearn.metrics import silhouette_score

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X = train.copy()
X_test = test.copy()

print("Train shape:", train.shape)
print("Test shape :", test.shape)


## Choosing the Number of Clusters (Elbow Method)

In [ ]:
# Sweep k and track Silhouette Score to pick a reasonable number of clusters.
# All models below use best_k instead of a hardcoded number.
k_range = range(2, 11)
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=0)
    labels = km.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))

plt.plot(list(k_range), sil_scores, marker="o")
plt.title("Silhouette Score vs k")
plt.xlabel("Number of clusters (k)")
plt.ylabel("Silhouette Score")
plt.show()

best_k = list(k_range)[int(np.argmax(sil_scores))]
print("Best k:", best_k)


## 1. K-Means Clustering

In [ ]:
from sklearn.cluster import KMeans

# n_clusters: number of clusters to form, taken from the elbow method above.
# n_init: number of times the algorithm runs with different centroid seeds; best result is kept.
model = KMeans(
    n_clusters=best_k,
    n_init=10,
    random_state=0
)

labels = model.fit_predict(X)

print("Train Silhouette Score:", silhouette_score(X, labels))


## 2. Hierarchical Clustering (Agglomerative, bottom-up)

In [ ]:
from sklearn.cluster import AgglomerativeClustering

# n_clusters: number of clusters to form, taken from the elbow method above.
# linkage: "ward" minimizes within-cluster variance (only works with euclidean distance).
# Agglomerative works bottom-up: every point starts as its own cluster and the
# closest pairs are merged step by step until n_clusters remain.
model = AgglomerativeClustering(
    n_clusters=best_k,
    linkage="ward"
)

labels = model.fit_predict(X)

print("Train Silhouette Score:", silhouette_score(X, labels))


## 3. Bisecting K-Means (Divisive, top-down)

In [ ]:
from sklearn.cluster import BisectingKMeans

# n_clusters: number of clusters to form, taken from the elbow method above.
# Bisecting K-Means works top-down: it starts with one big cluster and
# repeatedly splits the largest/worst cluster in two (using K-Means with k=2)
# until n_clusters remain. This is the top-down counterpart to Agglomerative's
# bottom-up approach. Scikit-learn does not offer a true divisive hierarchical
# algorithm, so this is the standard stand-in for it.
model = BisectingKMeans(
    n_clusters=best_k,
    random_state=0
)

labels = model.fit_predict(X)

print("Train Silhouette Score:", silhouette_score(X, labels))


## Final Test-Set Evaluation

In [ ]:
# IMPORTANT:
# Every model is fit on train.csv only.
# test.csv remains completely untouched until this cell.
# There is no target column, so Silhouette Score is used instead of accuracy/R2.

models = {
    "K-Means": KMeans(n_clusters=best_k, n_init=10, random_state=0),
    "Hierarchical (Agglomerative)": AgglomerativeClustering(n_clusters=best_k, linkage="ward"),
    "Bisecting K-Means (Divisive)": BisectingKMeans(n_clusters=best_k, random_state=0)
}

results = []

for name, model in models.items():

    train_labels = model.fit_predict(X)
    train_score = silhouette_score(X, train_labels)

    # KMeans and BisectingKMeans can predict on new data directly.
    # AgglomerativeClustering has no predict(), so it is refit on test.csv instead
    # to get test-cluster labels (results are still comparable via Silhouette Score).
    if hasattr(model, "predict"):
        test_labels = model.predict(X_test)
    else:
        test_labels = model.fit_predict(X_test)

    test_score = silhouette_score(X_test, test_labels)

    results.append([name, train_score, test_score])

results_df = pd.DataFrame(
    results,
    columns=["Model", "Train Silhouette Score", "Test Silhouette Score"]
)

results_df = results_df.sort_values(
    "Test Silhouette Score",
    ascending=False
).reset_index(drop=True)

display(results_df)

best_row = results_df.iloc[0]

print("Best model:", best_row["Model"])
print("Best test Silhouette Score:", best_row["Test Silhouette Score"])


## Save Best Clustering Model

The best clustering model is selected using the existing **Test Silhouette Score** comparison in this notebook.
For Agglomerative Clustering, the saved model is refit on the full training dataset because it does not provide a `predict()` method for new data.


In [ ]:
# ============================================================
# SAVE BEST CLUSTERING MODEL
# ============================================================
# Clustering has no target column and therefore no supervised
# accuracy/R² metric. The existing notebook comparison uses
# Silhouette Score.

import joblib
from IPython.display import display, FileLink

best_model_name = best_row["Model"]

# Retrieve the corresponding model object.
best_model = models[best_model_name]

# AgglomerativeClustering does not have predict().
# Refit it on the full training data before saving so the saved
# object represents the training dataset rather than the test set.
if best_model_name == "Hierarchical (Agglomerative)":
    best_model = AgglomerativeClustering(
        n_clusters=best_k,
        linkage="ward"
    )
    best_model.fit(X)

clustering_model_package = {
    "model": best_model,
    "feature_columns": list(X.columns),
    "task": "clustering",
    "selection_metric": "Test Silhouette Score",
    "n_clusters": int(best_k),
    "best_test_silhouette_score": float(
        best_row["Test Silhouette Score"]
    )
}

clustering_pkl_path = "best_clustering_model.pkl"

joblib.dump(
    clustering_model_package,
    clustering_pkl_path
)

print("=" * 70)
print("Best clustering model saved successfully.")
print("Model:", best_model_name)
print("File:", clustering_pkl_path)
print("Clusters:", best_k)
print("=" * 70)

display(FileLink(clustering_pkl_path, result_html_prefix="⬇️ Download Best Clustering Model: "))
